# Phase 5 - With/Without-DarkIR Comparison + Report

The project's actual deliverable: proves, with numbers, that brightening a
dark endoscope frame with DarkIR-lite *before* Mini-3D-Recon improves the
reconstruction, versus feeding the same dark frame straight in.

Both conditions run on the **same synthetically-degraded input** (the
dataset only ever stores clean frames -- `src/data/dark_degradation.py`
applies the degradation on the fly, seeded per dataset-window index so both
conditions see byte-identical dark frames -- see
`src/eval/run_comparison.py`'s module docstring) on the UnityCam **test**
split (held out from both Phase 2 and Phase 3 training).

- `raw_dark_input`: degraded frame -> Mini-3D-Recon directly.
- `darkir_lite_enhanced`: degraded frame -> DarkIR-lite (Phase 2 checkpoint)
  -> Mini-3D-Recon.

Metrics: depth AbsRel/RMSE/delta1 (median-ratio scaled -- depth units were
never confirmed, see PROGRESS.md) and trajectory ATE/RPE (Umeyama-aligned --
predicted pose translation is in raw, uncalibrated units). See
`src/eval/metrics.py` for the implementation and PROGRESS.md for why each
choice was made.

**GPU is off** -- both models are small and this is inference-only.

## 0. Setup: clone our repo + DarkIR upstream, install deps

In [ ]:
REPO_URL = "https://github.com/ritiksharma3/endoslam.git"
DARKIR_URL = "https://github.com/cidautai/DarkIR.git"

!git clone $REPO_URL repo
!git clone $DARKIR_URL repo/DarkIR_upstream
%cd repo

!pip install -q -r environment/requirements.txt  # includes ptflops (required just to import
                                                   # DarkIR's archs/ package) -- no torch reinstall,
                                                   # GPU is off so the P100/Pascal issue doesn't apply

import sys
sys.path.insert(0, ".")

## 1. Resolve dataset root + Phase 2 (DarkIR) + Phase 3 (Mini-3D-Recon) checkpoints

In [ ]:
import os

def find_endoslam_root(base="/kaggle/input", max_depth=4):
    for root, dirs, _files in os.walk(base):
        depth = root[len(base):].count(os.sep)
        if depth > max_depth:
            dirs[:] = []
            continue
        if os.path.basename(root).lower() == "endoslam":
            return root
    return None

def find_phase_checkpoint(name_fragment: str, base="/kaggle/input"):
    # Two kernel_sources are mounted this time (Phase 4 only needed one), each
    # under its own /kaggle/input/<slug>/... subtree -- both contain files
    # named epoch_*.pt, so a bare "search all of /kaggle/input" (Phase 4's
    # approach) would conflate them. Filtering by a name fragment unique to
    # each kernel's slug (checked against the containing directory path, not
    # hardcoded to an exact mount path) disambiguates without assuming the
    # exact mount layout.
    candidates = [os.path.join(r, f) for r, _, fs in os.walk(base) for f in fs
                  if f.startswith("epoch_") and f.endswith(".pt") and name_fragment in r.lower()]
    if not candidates:
        return None
    return max(candidates, key=lambda p: int(os.path.basename(p).split("_")[1].split(".")[0]))

DATA_ROOT = find_endoslam_root()
assert DATA_ROOT, "could not find an endoslam dir under /kaggle/input"
print("DATA_ROOT:", DATA_ROOT)

DARKIR_CHECKPOINT_PATH = find_phase_checkpoint("darkir")
assert DARKIR_CHECKPOINT_PATH, "could not find a Phase 2 DarkIR-lite checkpoint under /kaggle/input"
print("DARKIR_CHECKPOINT_PATH:", DARKIR_CHECKPOINT_PATH)

MINI_RECON_CHECKPOINT_PATH = find_phase_checkpoint("mini3drecon")
assert MINI_RECON_CHECKPOINT_PATH, "could not find a Phase 3 Mini-3D-Recon checkpoint under /kaggle/input"
print("MINI_RECON_CHECKPOINT_PATH:", MINI_RECON_CHECKPOINT_PATH)

## 2. Load config, test-split dataset, and both trained models

In [ ]:
import yaml
import torch

with open("configs/config.yaml") as f:
    config = yaml.safe_load(f)
config["data"]["root"] = DATA_ROOT

from src.common.device import select_device
from src.data.endoslam_dataset import EndoSLAMStomachDataset
from src.darkir_lite.model import build_darkir_lite
from src.reconstruction.model import MiniReconModel

device = select_device()
print("device:", device)

# test split -- held out from both Phase 2 (darkir_lite trained on train+val
# frames across all cameras) and Phase 3 (mini_recon trained on UnityCam
# train split only) -- first genuinely unseen evaluation of either model.
dataset = EndoSLAMStomachDataset(config, split="test", cameras=["UnityCam"],
                                  context_window=config["reconstruction"]["context_window"])
print(f"UnityCam test windows: {len(dataset)}")

mini_recon_model = MiniReconModel(pretrained=False, depth_head_channels=config["reconstruction"]["depth_head_channels"]).to(device)
mini_checkpoint = torch.load(MINI_RECON_CHECKPOINT_PATH, map_location=device, weights_only=False)
mini_recon_model.load_state_dict(mini_checkpoint["model_state_dict"])
mini_recon_model.eval()
print(f"loaded Mini-3D-Recon checkpoint: epoch={mini_checkpoint['epoch']}, "
      f"val_depth_absrel={mini_checkpoint.get('val_depth_absrel')}")

darkir_model = build_darkir_lite(pretrained=False).to(device)
darkir_checkpoint = torch.load(DARKIR_CHECKPOINT_PATH, map_location=device, weights_only=False)
darkir_model.load_state_dict(darkir_checkpoint["model_state_dict"])
darkir_model.eval()
print(f"loaded DarkIR-lite checkpoint: epoch={darkir_checkpoint['epoch']}, "
      f"val_psnr={darkir_checkpoint.get('val_psnr')}, val_ssim={darkir_checkpoint.get('val_ssim')}")

## 3. Run both conditions

`compare_conditions()` degrades every test-split frame once per condition,
seeded by dataset-window index (see `run_comparison.py`) so both conditions
see identical dark input -- the only difference is whether DarkIR-lite runs
in between.

In [ ]:
from src.eval.run_comparison import compare_conditions

results = compare_conditions(dataset, mini_recon_model, darkir_model, config, device)

for condition, r in results.items():
    print(f"\n--- {condition} ---")
    print("depth:   ", r["depth"])
    print("trajectory:", r["trajectory"])

## 4. Save metrics JSON + preview triplets

In [ ]:
import json

import matplotlib.pyplot as plt
import numpy as np

os.makedirs("/kaggle/working/previews", exist_ok=True)

metrics_summary = {cond: {"depth": r["depth"], "trajectory": r["trajectory"]} for cond, r in results.items()}
print(json.dumps(metrics_summary, indent=2))
with open("/kaggle/working/phase5_metrics.json", "w") as f:
    json.dump(metrics_summary, f, indent=2)
print("saved: /kaggle/working/phase5_metrics.json")


def save_preview_triplet(preview: dict, name: str):
    cols = [c for c in ("dark", "enhanced", "clean") if preview.get(c) is not None]
    fig, axes = plt.subplots(1, len(cols), figsize=(6 * len(cols), 6))
    if len(cols) == 1:
        axes = [axes]
    for ax, key in zip(axes, cols):
        ax.imshow(np.clip(preview[key], 0.0, 1.0))
        ax.set_title(key)
        ax.axis("off")
    plt.tight_layout()
    path = f"/kaggle/working/previews/{name}.png"
    plt.savefig(path, dpi=100)
    plt.close(fig)
    print("saved:", path)


for condition, r in results.items():
    for i, preview in enumerate(r["previews"]):
        save_preview_triplet(preview, f"{condition}_{i}")

## Done

Download `phase5_metrics.json` and `previews/*.png`. Compare `AbsRel` /
`RMSE` / `ATE` / `RPE_trans_rmse` / `RPE_rot_rmse_deg` between
`raw_dark_input` and `darkir_lite_enhanced` -- `darkir_lite_enhanced` should
come out lower across the board (that's the project's core hypothesis). If
it doesn't on some metric, that's a real finding for the report, not
something to explain away. Write `REPORT.md` at the repo root from these
numbers + a couple of the preview triplets, then log this run in
PROGRESS.md.